In [0]:
import subprocess
import sys
import re
import time
import os
import atexit
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pyspark import SparkConf
from pyspark import SparkContext
from pyspark import SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import concat, col, udf, lag, date_add, explode, lit, unix_timestamp, regexp_extract
from pyspark.sql.functions import month, weekofyear, dayofmonth, year, hour, minute, second, to_timestamp
from pyspark.sql.types import *
from pyspark.sql.types import DateType
from pyspark.sql.types import DataType
from pyspark.sql.window import Window
from pyspark.sql import Row
from pyspark.ml.classification import *
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler,OneHotEncoder,VectorIndexer, PCA, RFormula
from pyspark.ml import Pipeline, PipelineModel
from delta import DeltaTable

# Data Cleansing

In [0]:
df = spark.read.table("fraud_detection_project.bronze_layer.customer_profiles")

# Standardize column names
def StandardizeNames(df):
    l = df.columns                                                  # (Regex Operator -> https://regex101.com/)
    cols = [re.sub(r'(?<!^)(?=[A-Z])', '_', c).lower() for c in l]  # Convert CamelCase to snake_case
    cols = [c.lstrip('_') for c in cols]  # Remove underscores in the beginning of column names
    return df.toDF(*cols)
df = StandardizeNames(df)

In [0]:
# Deleting duplicated data
df.dropDuplicates(['customer_id'])

# Deleting rows without some features
df = df.dropna(how='any', subset=['customer_id','file_path','ingest_datetime'])

In [0]:
# Extracting card type from client_details column
df = df.withColumn("card_type", regexp_extract(col("client_details"), r"Type:([^|]+)", 1))

# Extracting card bin from client_details column
df = df.withColumn("card_bin", regexp_extract(col("client_details"), r"BIN:(\d+)",1))

# Deleting client_details column
df.drop("client_details")

In [0]:
df.show(1)

# Feature Engineering